In [ ]:
import random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def gerar_dataset_sintetico(
    data_inicial='2024-01-01',
    data_final='2024-12-31',
    numero_clientes=200,
    transacoes_diarias_min=30,
    transacoes_diarias_max=80,
    seed=42
):

    random.seed(seed)
    np.random.seed(seed)

    # Dicionário de categorias e produtos
    categorias_produtos = {
        'Hortifruti': ['Banana', 'Maçã', 'Tomate', 'Alface', 'Cenoura', 'Batata', 'Cebola', 'Laranja', 'Pimentão', 'Abobrinha'],
        'Laticínios': ['Leite', 'Iogurte', 'Queijo Mussarela', 'Queijo Prato', 'Manteiga', 'Requeijão', 'Creme de Leite', 'Chantilly'],
        'Carnes': ['Carne Bovina', 'Frango', 'Peito de Peru', 'Linguiça', 'Peixe', 'Carne Suína'],
        'Padaria': ['Pão de Forma', 'Pão Francês', 'Bolo Simples', 'Croissant', 'Bolacha', 'Panetone'],
        'Bebidas': ['Água Mineral', 'Refrigerante', 'Suco', 'Cerveja', 'Vinho', 'Café', 'Chá'],
        'Snacks': ['Batata Frita (salgadinho)', 'Amendoim', 'Chocolate', 'Biscoito Recheado', 'Barrinha de Cereais'],
        'Limpeza': ['Sabão em Pó', 'Detergente', 'Desinfetante', 'Água Sanitária', 'Sabão em Barra'],
        'Higiene Pessoal': ['Sabonete', 'Shampoo', 'Condicionador', 'Pasta de Dente', 'Escova de Dentes'],
        'Mercearia': ['Arroz', 'Feijão', 'Açúcar', 'Macarrão', 'Farinha de Trigo', 'Óleo de Soja'],
        'Congelados': ['Pizza Congelada', 'Hambúrguer Congelado', 'Lasanha Congelada']
    }

    # Lista de feriados/datas festivas para criar picos de compras específicas
    # (Ajuste conforme o ano real e as datas que desejar simular)
    datas_festivas = {
        'Páscoa':  ('2024-03-31', ['Chocolate', 'Ovo de Páscoa']),
        'Dia das Mães': ('2024-05-12', ['Bolo Simples', 'Chocolate', 'Vinho']),
        'Dia dos Pais': ('2024-08-11', ['Cerveja', 'Linguiça', 'Carne Bovina']),
        'Natal':   ('2024-12-25', ['Panetone', 'Carne Bovina', 'Vinho']),
        'Ano Novo':('2024-12-31', ['Vinho', 'Cerveja'])
    }

    # Simulação de promoções ocasionais por categoria em determinadas datas
    # A chave é (mês, categoria) e o valor é o desconto (por exemplo, 10% = 0.10)
    promocoes_mensais = {
        (1, 'Hortifruti'): 0.10,  # Ex.: Em janeiro, desconto em hortifruti
        (6, 'Bebidas'): 0.15,     # Em junho, desconto em bebidas
        (11, 'Carnes'): 0.20      # Em novembro, desconto em carnes
    }

    # Correlações de compra (itens frequentemente adquiridos juntos)
    # A chave é um item, e o valor é uma lista de itens que frequentemente aparecem juntos
    correlacoes_itens = {
        'Leite': ['Café', 'Pão de Forma'],
        'Cerveja': ['Linguiça', 'Amendoim'],
        'Arroz': ['Feijão', 'Óleo de Soja'],
        'Chocolate': ['Iogurte', 'Biscoito Recheado'],
        'Pão Francês': ['Manteiga', 'Queijo Mussarela']
    }

    # Convertendo strings de data para datetime
    data_inicial = datetime.strptime(data_inicial, '%Y-%m-%d')
    data_final = datetime.strptime(data_final, '%Y-%m-%d')

    # Função auxiliar para sortear produtos de categorias aleatórias
    def sortear_produtos():
        # Escolhe quantas categorias serão compradas (1 a 4, por exemplo)
        num_categorias = random.randint(1, 4)
        # Escolhe as categorias
        categorias_escolhidas = random.sample(list(categorias_produtos.keys()), num_categorias)

        produtos_compra = []
        for cat in categorias_escolhidas:
            # Escolhe quantos produtos dessa categoria (1 a 3)
            num_prod_cat = random.randint(1, 3)
            produtos_escolhidos = random.sample(categorias_produtos[cat], num_prod_cat)
            produtos_compra.extend(produtos_escolhidos)

        # Verifica se algum item puxa correlacionados
        adicionais = []
        for item in produtos_compra:
            if item in correlacoes_itens:
                # Há chance de 50% de adicionar os itens correlacionados
                if random.random() < 0.50:
                    correlatos = correlacoes_itens[item]
                    # Adiciona 1 ou 2 itens correlacionados aleatórios
                    qtd_correlatos = random.randint(1, min(2, len(correlatos)))
                    adicionais.extend(random.sample(correlatos, qtd_correlatos))

        produtos_compra.extend(adicionais)

        # Remove duplicados (caso tenha se repetido)
        produtos_compra = list(set(produtos_compra))
        return produtos_compra

    # Função auxiliar para obter categorias de uma lista de produtos
    def obter_categorias(produtos):
        categorias = []
        for cat, lista in categorias_produtos.items():
            # Se algum produto da compra estiver nessa lista, associamos
            intersecao = set(produtos).intersection(set(lista))
            if intersecao:
                categorias.append(cat)
        return list(set(categorias))

    # Geração de datas diárias dentro do intervalo
    dias = (data_final - data_inicial).days + 1
    datas = [data_inicial + timedelta(days=i) for i in range(dias)]

    registros = []

    for dia in datas:
        # Número de transações naquele dia
        n_transacoes_dia = random.randint(transacoes_diarias_min, transacoes_diarias_max)

        # Possibilidade de aumentar as transações em datas festivas (ex.: duplicar)
        for festivo, (data_fest, itens_festivos) in datas_festivas.items():
            fest_date = datetime.strptime(data_fest, '%Y-%m-%d')
            if dia.date() == fest_date.date():
                n_transacoes_dia *= 2  # dobra as transações no dia festivo

        for _ in range(n_transacoes_dia):
            # Escolhe um horário aleatório do dia (entre 8h e 22h)
            hora_random = random.randint(8, 22)
            minuto_random = random.randint(0, 59)
            data_hora_transacao = dia + timedelta(hours=hora_random, minutes=minuto_random)

            # Sorteia um cliente
            id_cliente = f"C{random.randint(1, numero_clientes):03d}"

            # Sorteia produtos
            produtos = sortear_produtos()

            # Verifica se há itens festivos e adiciona caso tenha chance em datas especiais
            # (por exemplo, em datas específicas, maior probabilidade de itens temáticos)
            for festivo, (data_fest, itens_festivos) in datas_festivas.items():
                fest_date = datetime.strptime(data_fest, '%Y-%m-%d')
                if dia.date() == fest_date.date():
                    # Para cada item festivo, há 50% de chance de entrar
                    for item_fest in itens_festivos:
                        if random.random() < 0.50:
                            produtos.append(item_fest)
                    produtos = list(set(produtos))  # remove duplicados

            # Identifica categorias envolvidas
            lista_categorias = obter_categorias(produtos)

            # Calcula valor total (cada produto com um valor base + pequeno ruído)
            # Aqui, definimos valores médios por categoria de forma simplificada
            valor_base_cat = {
                'Hortifruti': 5.0,
                'Laticínios': 8.0,
                'Carnes': 15.0,
                'Padaria': 6.0,
                'Bebidas': 7.0,
                'Snacks': 4.0,
                'Limpeza': 10.0,
                'Higiene Pessoal': 8.0,
                'Mercearia': 5.0,
                'Congelados': 12.0
            }

            valor_total = 0.0
            for prod in produtos:
                # Encontra categoria desse produto (a primeira correspondente)
                cat_prod = None
                for cat, lista_prod in categorias_produtos.items():
                    if prod in lista_prod:
                        cat_prod = cat
                        break
                if cat_prod is not None:
                    # Adiciona um valor médio mais um ruído
                    valor_total += valor_base_cat[cat_prod] + random.uniform(-1, 1)
                else:
                    # Caso não encontre, adiciona valor genérico
                    valor_total += 5.0

            # Aplica promoções mensais por categoria
            desconto = 0.0
            mes_transacao = data_hora_transacao.month
            for cat in lista_categorias:
                if (mes_transacao, cat) in promocoes_mensais:
                    desconto_cat = promocoes_mensais[(mes_transacao, cat)]
                    desconto += desconto_cat  # soma percentual de desconto

            # Garante que o desconto máximo não ultrapasse 30%, por exemplo
            if desconto > 0.30:
                desconto = 0.30
            valor_total = valor_total * (1 - desconto)

            # Arredonda valor total
            valor_total = round(valor_total, 2)

            registros.append({
                'data_hora': data_hora_transacao.strftime('%Y-%m-%d %H:%M'),
                'id_cliente': id_cliente,
                'lista_produtos': ';'.join(sorted(produtos)),
                'categorias_produtos': ';'.join(sorted(lista_categorias)),
                'valor_total': valor_total
            })

    df = pd.DataFrame(registros)
    # Embaralha todo o DataFrame para que as transações não fiquem em ordem estritamente cronológica
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    return df

if __name__ == "__main__":
    df_transacoes = gerar_dataset_sintetico(
        data_inicial='2024-01-01',
        data_final='2024-12-31',
        numero_clientes=200,
        transacoes_diarias_min=30,
        transacoes_diarias_max=80,
        seed=42
    )

    # Salva em CSV
    df_transacoes.to_csv('transacoes_supermercado_sintetico.csv', index=False, encoding='utf-8-sig')

    # Exibe as 10 primeiras linhas como exemplo
    print(df_transacoes.head(10))
